In [11]:
import opt_einsum as oe
import numpy as np
import torch
import sys
sys.path.append("../../")
import mps
from mps.trainer.data_utils import create_mnist_dataloader, SyntheticDataset, SyntheticDatasetV2, SyntheticDatasetV3

In [12]:
N = 500
dataset = SyntheticDatasetV2(n = N, num_samples=2**15, seed = 0)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=2**12, shuffle=True)


In [13]:
from mps.trainer.smps_trainer import smps_train
from mps.simple_mps import SimpleMPS
import copy
# smps_params = torch.load("smps_2000.pth", weights_only=False)

chi = 2
d = 2
l = 2
device = torch.device("cpu")
dtype = torch.float64
optimize = "greedy"
eps = 1 / N
smps = SimpleMPS(N, chi, d, l, layers=1, device=device, dtype=dtype, optimize=optimize, eps=eps)

# smps.mps.params[0].data[:] = torch.tensor([ [0, 1], [1, 0] ])
# for i in range(1, len(smps.mps.params) - 1):
#     smps.mps.params[1].data[:] = torch.stack((torch.eye(2), ) * 2).permute(1, 0, 2)

Path is not set, setting...
Found the path
Initialized MPS with random matrices


In [14]:
from mps.trainer.smps_trainer import smps_train
import copy

epochs = 100
lr = 0.00001
logsoftmax = torch.nn.LogSoftmax(dim=-1)
nnloss = torch.nn.NLLLoss(reduction="mean")
opt_smps = torch.optim.Adam(smps.parameters(), lr=lr)
smps_losses = []
smps.train()
print(f"\n=== Training SimpleMPS for {epochs} epoch(s)... ===")
for epoch in range(epochs):
    total_loss = 0.0
    total_samples = 0
    total_correct = 0
    for batch_idx, (data, target) in enumerate(dataloader):
        data, target = data.to(device), target.to(device)
        data = data.permute(1, 0, 2)  # [batch, N, 2] → [N, batch, 2]
        opt_smps.zero_grad()
        outputs = smps(data)
        outputs = torch.abs(outputs)
        outputs = logsoftmax(outputs)
        loss = nnloss(outputs, target)
        loss.backward()
        
        # Print the norm of the gradients for 10 equally split indices
        params = list(smps.parameters())
        num_params = len(params)
        indices = [int(i * num_params / 10) for i in range(10)]
        grad_norms = [f"Gradient norm for parameter {idx}: {params[idx].grad.norm().item():.6f}" for idx in indices if params[idx].grad is not None]
        print(" | ".join(grad_norms))
        
        opt_smps.step()
        bs = target.size(0)
        total_loss += loss.item() * bs
        total_samples += bs
        preds = outputs.argmax(dim=-1)
        acc = (preds == target).float().sum().item()
        total_correct += acc
        print(f"[SimpleMPS] Epoch {epoch+1}, Step {batch_idx+1}/{len(dataloader)} | Loss: {loss.item():.6f} | Acc: {acc/bs:.2%}")
    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples
    smps_losses.append(epoch_loss)
    print(f"[SimpleMPS] Epoch {epoch+1} | Loss: {epoch_loss:.6f} | Acc: {epoch_acc:.2%}")


=== Training SimpleMPS for 100 epoch(s)... ===
Gradient norm for parameter 0: 0.007651 | Gradient norm for parameter 50: 0.011563 | Gradient norm for parameter 100: 0.010708 | Gradient norm for parameter 150: 0.010791 | Gradient norm for parameter 200: 0.011479 | Gradient norm for parameter 250: 0.011880 | Gradient norm for parameter 300: 0.010866 | Gradient norm for parameter 350: 0.012379 | Gradient norm for parameter 400: 0.023127 | Gradient norm for parameter 450: 0.011993
[SimpleMPS] Epoch 1, Step 1/8 | Loss: 0.693347 | Acc: 50.59%
Gradient norm for parameter 0: 0.018271 | Gradient norm for parameter 50: 0.011241 | Gradient norm for parameter 100: 0.014066 | Gradient norm for parameter 150: 0.008486 | Gradient norm for parameter 200: 0.008439 | Gradient norm for parameter 250: 0.004778 | Gradient norm for parameter 300: 0.019104 | Gradient norm for parameter 350: 0.010341 | Gradient norm for parameter 400: 0.006185 | Gradient norm for parameter 450: 0.012909
[SimpleMPS] Epoch 1, 

In [10]:
import os
# Define the file path for storing epoch metrics
epoch_metrics_file_path = "epoch_metrics_1024.csv"
# Define the file path for storing iteration metrics
iter_metrics_file_path = "iter_metrics_1024.csv"

# Initialize the epoch metrics file with headers if it doesn't exist
if not os.path.exists(epoch_metrics_file_path):
    print("Epoch metrics file does not exist. Creating it.")
    with open(epoch_metrics_file_path, 'w') as f:
        f.write("epoch,avg_loss,avg_loss_with_reg,avg_acc,avg_srpq,avg_reg\n")

# Initialize the iteration metrics file with headers if it doesn't exist
if not os.path.exists(iter_metrics_file_path):
    print("Iteration metrics file does not exist. Creating it.")
    with open(iter_metrics_file_path, 'w') as f:
        f.write("epoch,iter,loss0,reg,loss,acc,srpq\n")

In [11]:
from mps import tpcp_mps
from mps.trainer.utils import calculate_accuracy, mean_risk, loss_batch
# --- Step 2: Build and Prepare TPCP ---
tpcp = tpcp_mps.MPSTPCP(
    N,
    K=1,
    d=2,
    enable_r=True,
    with_identity=False,
    manifold=tpcp_mps.ManifoldType.EXACT,
)
tpcp.train()
# tpcp.set_canonical_mps(smps)

# torch.save(tpcp.state_dict(), "tpcp_1024_w_0_0.pth")

# state = torch.load("tpcp_state_dict_first.pth")
# tpcp.load_state_dict(state)

MPSTPCP(
  (kraus_ops): kraus_operators(
    (manifold): Stiefel(euclidean)(exact) manifold
    (kraus_ops): ParameterList(
        (0): Parameter containing: [torch.float64 of size 4x4]
        (1): Parameter containing: [torch.float64 of size 4x4]
        (2): Parameter containing: [torch.float64 of size 4x4]
        (3): Parameter containing: [torch.float64 of size 4x4]
        (4): Parameter containing: [torch.float64 of size 4x4]
        (5): Parameter containing: [torch.float64 of size 4x4]
        (6): Parameter containing: [torch.float64 of size 4x4]
        (7): Parameter containing: [torch.float64 of size 4x4]
        (8): Parameter containing: [torch.float64 of size 4x4]
        (9): Parameter containing: [torch.float64 of size 4x4]
        (10): Parameter containing: [torch.float64 of size 4x4]
        (11): Parameter containing: [torch.float64 of size 4x4]
        (12): Parameter containing: [torch.float64 of size 4x4]
        (13): Parameter containing: [torch.float64 of 

In [9]:
W = torch.zeros(tpcp.L, 2, dtype=torch.float64)
W[:, 0] = 1 
W[:, 1] = 0.002
tpcp.initialize_W(W)
# W_now = tpcp.W.clone()
# W_new = update_weights(W_now, 0.01)
# tpcp.initialize_W(W_new)
# torch.save(tpcp.state_dict(), "tpcp_state_dict_new.pth")
# 
# --- Step 3: Determine lambda_final Using the Initial Loss Value ---
data_batch, target_batch = next(iter(dataloader))
initial_probs, reg = tpcp(data_batch, return_probs=True, return_reg=True)
initial_accuracy = calculate_accuracy(initial_probs[:, 0], target_batch)
srpq = torch.exp(-reg)
sr_sys = torch.exp(-reg*N)
loss = loss_batch(initial_probs, target_batch)
print("srpq: ", srpq)
print("success rate : ", sr_sys)
print(f"Initial accuracy: {initial_accuracy.item():.2%}")
print(f"Initial loss: {loss.item()}")

srpq:  tensor(0.5010, dtype=torch.float64, grad_fn=<ExpBackward0>)
success rate :  tensor(4.3003e-308, dtype=torch.float64, grad_fn=<ExpBackward0>)
Initial accuracy: 100.00%
Initial loss: 0.28684051626649687


In [54]:
from mps.trainer.adaptive_mpsae_trainer import RiemannianAdam
from geoopt import optim
from mps.StiefelOptimizers import StiefelAdam, StiefelSGD
import torch
lr = 0.001
# optimizer = StiefelAdam(tpcp.kraus_ops.parameters(), lr=lr)
# optimizer = optim.RiemannianAdam(tpcp.kraus_ops.parameters(), lr=lr, betas=(0.99, 0.999))
optimizer = RiemannianAdam(tpcp.kraus_ops.parameters(), lr=lr)
optimizer_weight = torch.optim.Adam([tpcp.r, tpcp.W], lr=0.05*lr)
tpcp.W.requires_grad = True
tpcp.r.requires_grad = True

In [15]:
import pandas as pd
import os
from mps.trainer.utils import calculate_accuracy, mean_risk, loss_batch
from mps.trainer.adaptive_mpsae_trainer import RiemannianAdam

clambda = 100

def set_W_min(W, min_value):
    W = W.clone()
    return torch.maximum(W, torch.tensor(min_value))


def train_tpcp(tpcp_params, dataloader, lr, clambda, w_value, train_W=True):

    epochs = 100
    tpcp = tpcp_mps.MPSTPCP(
        N,
        K=1,
        d=2,
        enable_r=True,
        with_identity=False,
        manifold=tpcp_mps.ManifoldType.EXACT,
    )
    tpcp.load_state_dict(tpcp_params)
    W = set_W_min(tpcp.W, w_value)
    tpcp.initialize_W(W)
    optimizer = RiemannianAdam(tpcp.kraus_ops.parameters(), lr=lr)
    optimizer_weight = torch.optim.Adam([tpcp.r, tpcp.W], lr=0.05*lr)
    tpcp.W.requires_grad = train_W
    tpcp.r.requires_grad = True

    for epoch in range(epochs):

        for iter_num, (data, target) in enumerate(dataloader, start=1):
            data_flipped = data.clone()
            data_flipped[:, 0] = 1 - data_flipped[:, 0]  # Flip the first element in each data in the batch
            target_flipped = 1 - target  # Flip the target

            data = torch.cat((data, data_flipped), dim=0)
            target = torch.cat((target, target_flipped), dim=0)

            optimizer.zero_grad()
            optimizer_weight.zero_grad()
            outputs, reg = tpcp(data, return_probs=True, return_reg=True)
            loss0 = loss_batch(outputs, target, gamma=0.0, alpha = 0.5)
            risk = mean_risk(outputs, target)
            # ls = logsoftmax(outputs)
            # loss0 = nnloss(ls, target)
            loss = loss0 + clambda * reg

            loss.backward()

            optimizer.step()
            optimizer_weight.step()

            tpcp.proj_stiefel(check_on_manifold=True, print_log=False, rtol=1e-3)
            tpcp.normalize_w_and_r()

            acc = calculate_accuracy(outputs[:, 0], target)
            srpq = torch.exp(-reg)


            # Log iteration metrics
            print(f"Epoch {epoch+1}, Iter {iter_num}, Loss0: {loss0.item():.6f}, reg: {reg.item():.6f}, Loss: {loss.item():.6f}, Acc: {acc.item():.2%}, SRPQ: {srpq.item():.6e}, Risk: {risk.item():.6f}")

            # Write iteration metrics to file
            with open(iter_metrics_file_path, 'a') as f:
                f.write(f"{epoch+1},{iter_num},{loss0.item():.6f},{reg.item():.6f},{loss.item():.6f},{acc.item():.6f},{srpq.item():.6e},{w_value}\n")

            torch.save(tpcp.state_dict(), "tpcp_1024_w_{}.pth".format(w_value))

In [22]:
params = torch.load("tpcp_1024_w_0_0.pth")
w_value = 0.002
lr = 0.001
clambda = 70

train_tpcp(params, dataloader, lr, clambda, w_value)




Epoch 1, Iter 1, Loss0: 0.287271, reg: 0.691151, Loss: 48.667874, Acc: 100.00%, SRPQ: 5.009989e-01, Risk: 0.436917
Epoch 1, Iter 2, Loss0: 0.302695, reg: 0.691131, Loss: 48.681867, Acc: 100.00%, SRPQ: 5.010091e-01, Risk: 0.453996
Epoch 1, Iter 3, Loss0: 0.301496, reg: 0.691105, Loss: 48.678870, Acc: 100.00%, SRPQ: 5.010220e-01, Risk: 0.452710
Epoch 1, Iter 4, Loss0: 0.299754, reg: 0.691086, Loss: 48.675777, Acc: 100.00%, SRPQ: 5.010316e-01, Risk: 0.450807
Epoch 1, Iter 5, Loss0: 0.287721, reg: 0.691066, Loss: 48.662365, Acc: 100.00%, SRPQ: 5.010415e-01, Risk: 0.437451
Epoch 1, Iter 6, Loss0: 0.295386, reg: 0.691048, Loss: 48.668740, Acc: 100.00%, SRPQ: 5.010507e-01, Risk: 0.445998
Epoch 1, Iter 7, Loss0: 0.295269, reg: 0.691046, Loss: 48.668495, Acc: 100.00%, SRPQ: 5.010517e-01, Risk: 0.445869
Epoch 1, Iter 8, Loss0: 0.287321, reg: 0.691033, Loss: 48.659628, Acc: 100.00%, SRPQ: 5.010582e-01, Risk: 0.437004
Epoch 1, Iter 9, Loss0: 0.286980, reg: 0.691019, Loss: 48.658291, Acc: 100.00%, 

KeyboardInterrupt: 

In [43]:
params["W"][:, 1]

tensor([6.5206e-05, 4.7234e-05, 1.0790e-03,  ..., 4.4755e-05, 2.2758e-05,
        1.8810e-05], dtype=torch.float64)

In [47]:
params = torch.load("tpcp_1024_w_0.002.pth")
w_value = 0.003
lr = 0.0005
clambda = 45

train_tpcp(params, dataloader, lr, clambda, w_value)

Epoch 1, Iter 1, Loss0: 0.321189, reg: 0.688379, Loss: 31.298240, Acc: 100.00%, SRPQ: 5.023898e-01, Risk: 0.473926
Epoch 1, Iter 2, Loss0: 0.322957, reg: 0.688384, Loss: 31.300241, Acc: 100.00%, SRPQ: 5.023872e-01, Risk: 0.475784
Epoch 1, Iter 3, Loss0: 0.320967, reg: 0.688370, Loss: 31.297610, Acc: 100.00%, SRPQ: 5.023944e-01, Risk: 0.473693
Epoch 1, Iter 4, Loss0: 0.320352, reg: 0.688382, Loss: 31.297536, Acc: 100.00%, SRPQ: 5.023883e-01, Risk: 0.473047
Epoch 1, Iter 5, Loss0: 0.319712, reg: 0.688378, Loss: 31.296731, Acc: 100.00%, SRPQ: 5.023902e-01, Risk: 0.472373
Epoch 1, Iter 6, Loss0: 0.319068, reg: 0.688373, Loss: 31.295844, Acc: 100.00%, SRPQ: 5.023929e-01, Risk: 0.471694
Epoch 1, Iter 7, Loss0: 0.318582, reg: 0.688390, Loss: 31.296114, Acc: 100.00%, SRPQ: 5.023845e-01, Risk: 0.471180
Epoch 1, Iter 8, Loss0: 0.317746, reg: 0.688382, Loss: 31.294944, Acc: 100.00%, SRPQ: 5.023882e-01, Risk: 0.470296
Epoch 1, Iter 9, Loss0: 0.316720, reg: 0.688374, Loss: 31.293553, Acc: 100.00%, 

KeyboardInterrupt: 

In [55]:
params = torch.load("tpcp_1024_w_0.0033.pth")
w_value = 0.0033
lr = 0.001
clambda = 43

train_tpcp(params, dataloader, lr, clambda, w_value)

Epoch 1, Iter 1, Loss0: 0.319246, reg: 0.686222, Loss: 29.826776, Acc: 100.00%, SRPQ: 5.034748e-01, Risk: 0.471911
Epoch 1, Iter 2, Loss0: 0.321968, reg: 0.686220, Loss: 29.829423, Acc: 100.00%, SRPQ: 5.034757e-01, Risk: 0.474778
Epoch 1, Iter 3, Loss0: 0.320248, reg: 0.686205, Loss: 29.827050, Acc: 100.00%, SRPQ: 5.034833e-01, Risk: 0.472969
Epoch 1, Iter 4, Loss0: 0.319432, reg: 0.686187, Loss: 29.825491, Acc: 100.00%, SRPQ: 5.034920e-01, Risk: 0.472108
Epoch 1, Iter 5, Loss0: 0.317723, reg: 0.686164, Loss: 29.822760, Acc: 100.00%, SRPQ: 5.035040e-01, Risk: 0.470300
Epoch 1, Iter 6, Loss0: 0.317665, reg: 0.686140, Loss: 29.821664, Acc: 100.00%, SRPQ: 5.035161e-01, Risk: 0.470239
Epoch 1, Iter 7, Loss0: 0.317619, reg: 0.686142, Loss: 29.821741, Acc: 100.00%, SRPQ: 5.035147e-01, Risk: 0.470190
Epoch 1, Iter 8, Loss0: 0.316013, reg: 0.686152, Loss: 29.820538, Acc: 100.00%, SRPQ: 5.035100e-01, Risk: 0.468485
Epoch 1, Iter 9, Loss0: 0.314189, reg: 0.686111, Loss: 29.816956, Acc: 100.00%, 

KeyboardInterrupt: 

In [57]:
params = torch.load("tpcp_1024_w_0.0033.pth")
w_value = 0.0038
lr = 0.001
clambda = 40

train_tpcp(params, dataloader, lr, clambda, w_value)

Epoch 1, Iter 1, Loss0: 0.319564, reg: 0.683302, Loss: 27.651661, Acc: 100.00%, SRPQ: 5.049467e-01, Risk: 0.472242
Epoch 1, Iter 2, Loss0: 0.319690, reg: 0.683283, Loss: 27.651026, Acc: 100.00%, SRPQ: 5.049563e-01, Risk: 0.472370
Epoch 1, Iter 3, Loss0: 0.317927, reg: 0.683280, Loss: 27.649142, Acc: 100.00%, SRPQ: 5.049578e-01, Risk: 0.470510
Epoch 1, Iter 4, Loss0: 0.317105, reg: 0.683274, Loss: 27.648062, Acc: 100.00%, SRPQ: 5.049611e-01, Risk: 0.469640
Epoch 1, Iter 5, Loss0: 0.315990, reg: 0.683268, Loss: 27.646712, Acc: 100.00%, SRPQ: 5.049640e-01, Risk: 0.468455
Epoch 1, Iter 6, Loss0: 0.314912, reg: 0.683252, Loss: 27.645007, Acc: 100.00%, SRPQ: 5.049720e-01, Risk: 0.467306
Epoch 1, Iter 7, Loss0: 0.313855, reg: 0.683232, Loss: 27.643147, Acc: 100.00%, SRPQ: 5.049821e-01, Risk: 0.466180
Epoch 1, Iter 8, Loss0: 0.312768, reg: 0.683226, Loss: 27.641809, Acc: 100.00%, SRPQ: 5.049853e-01, Risk: 0.465020
Epoch 1, Iter 9, Loss0: 0.311604, reg: 0.683221, Loss: 27.640446, Acc: 100.00%, 

KeyboardInterrupt: 

In [58]:
params = torch.load("tpcp_1024_w_0.0038.pth")
w_value = 0.0042
lr = 0.001
clambda = 35

train_tpcp(params, dataloader, lr, clambda, w_value)

Epoch 1, Iter 1, Loss0: 0.309874, reg: 0.681240, Loss: 24.153277, Acc: 100.00%, SRPQ: 5.059891e-01, Risk: 0.461920
Epoch 1, Iter 2, Loss0: 0.309925, reg: 0.681234, Loss: 24.153122, Acc: 100.00%, SRPQ: 5.059921e-01, Risk: 0.461969
Epoch 1, Iter 3, Loss0: 0.307647, reg: 0.681222, Loss: 24.150407, Acc: 100.00%, SRPQ: 5.059984e-01, Risk: 0.459514
Epoch 1, Iter 4, Loss0: 0.306042, reg: 0.681226, Loss: 24.148958, Acc: 100.00%, SRPQ: 5.059962e-01, Risk: 0.457778
Epoch 1, Iter 5, Loss0: 0.304762, reg: 0.681227, Loss: 24.147714, Acc: 100.00%, SRPQ: 5.059956e-01, Risk: 0.456388
Epoch 1, Iter 6, Loss0: 0.303538, reg: 0.681215, Loss: 24.146055, Acc: 100.00%, SRPQ: 5.060019e-01, Risk: 0.455055
Epoch 1, Iter 7, Loss0: 0.301974, reg: 0.681219, Loss: 24.144648, Acc: 100.00%, SRPQ: 5.059997e-01, Risk: 0.453348
Epoch 1, Iter 8, Loss0: 0.300184, reg: 0.681196, Loss: 24.142060, Acc: 100.00%, SRPQ: 5.060112e-01, Risk: 0.451387
Epoch 1, Iter 9, Loss0: 0.298587, reg: 0.681188, Loss: 24.140179, Acc: 100.00%, 

KeyboardInterrupt: 

In [59]:
params = torch.load("tpcp_1024_w_0.0042.pth")
w_value = 0.005
lr = 0.001
clambda = 35

train_tpcp(params, dataloader, lr, clambda, w_value)

Epoch 1, Iter 1, Loss0: 0.294045, reg: 0.674899, Loss: 23.915511, Acc: 100.00%, SRPQ: 5.092078e-01, Risk: 0.444584
Epoch 1, Iter 2, Loss0: 0.292166, reg: 0.674878, Loss: 23.912889, Acc: 100.00%, SRPQ: 5.092186e-01, Risk: 0.442488
Epoch 1, Iter 3, Loss0: 0.290170, reg: 0.674879, Loss: 23.910943, Acc: 100.00%, SRPQ: 5.092179e-01, Risk: 0.440263
Epoch 1, Iter 4, Loss0: 0.288333, reg: 0.674866, Loss: 23.908631, Acc: 100.00%, SRPQ: 5.092248e-01, Risk: 0.438204
Epoch 1, Iter 5, Loss0: 0.286560, reg: 0.674860, Loss: 23.906672, Acc: 100.00%, SRPQ: 5.092275e-01, Risk: 0.436208
Epoch 1, Iter 6, Loss0: 0.284653, reg: 0.674855, Loss: 23.904574, Acc: 100.00%, SRPQ: 5.092303e-01, Risk: 0.434053
Epoch 1, Iter 7, Loss0: 0.282793, reg: 0.674846, Loss: 23.902388, Acc: 100.00%, SRPQ: 5.092351e-01, Risk: 0.431945
Epoch 1, Iter 8, Loss0: 0.280955, reg: 0.674830, Loss: 23.899988, Acc: 100.00%, SRPQ: 5.092432e-01, Risk: 0.429854
Epoch 1, Iter 9, Loss0: 0.279034, reg: 0.674826, Loss: 23.897949, Acc: 100.00%, 

KeyboardInterrupt: 

In [62]:
params = torch.load("tpcp_1024_w_0.005.pth")
w_value = 0.0075
lr = 0.001
clambda = 35

train_tpcp(params, dataloader, lr, clambda, w_value)

Epoch 1, Iter 1, Loss0: 0.285831, reg: 0.671611, Loss: 23.792199, Acc: 100.00%, SRPQ: 5.108851e-01, Risk: 0.435410
Epoch 1, Iter 2, Loss0: 0.284179, reg: 0.671576, Loss: 23.789328, Acc: 100.00%, SRPQ: 5.109029e-01, Risk: 0.433534
Epoch 1, Iter 3, Loss0: 0.282505, reg: 0.671552, Loss: 23.786814, Acc: 100.00%, SRPQ: 5.109152e-01, Risk: 0.431636
Epoch 1, Iter 4, Loss0: 0.281121, reg: 0.671522, Loss: 23.784390, Acc: 100.00%, SRPQ: 5.109304e-01, Risk: 0.430063
Epoch 1, Iter 5, Loss0: 0.279673, reg: 0.671507, Loss: 23.782418, Acc: 100.00%, SRPQ: 5.109380e-01, Risk: 0.428413
Epoch 1, Iter 6, Loss0: 0.278320, reg: 0.671474, Loss: 23.779918, Acc: 100.00%, SRPQ: 5.109548e-01, Risk: 0.426864
Epoch 1, Iter 7, Loss0: 0.276927, reg: 0.671452, Loss: 23.777732, Acc: 100.00%, SRPQ: 5.109663e-01, Risk: 0.425265
Epoch 1, Iter 8, Loss0: 0.275465, reg: 0.671414, Loss: 23.774960, Acc: 100.00%, SRPQ: 5.109855e-01, Risk: 0.423581
Epoch 1, Iter 9, Loss0: 0.274029, reg: 0.671391, Loss: 23.772700, Acc: 100.00%, 

KeyboardInterrupt: 

In [65]:
params = torch.load("tpcp_1024_w_0.0075.pth")
w_value = 0.02
lr = 0.001
clambda = 35

train_tpcp(params, dataloader, lr, clambda, w_value)

Epoch 1, Iter 1, Loss0: 0.274036, reg: 0.663139, Loss: 23.483896, Acc: 100.00%, SRPQ: 5.152316e-01, Risk: 0.421937
Epoch 1, Iter 2, Loss0: 0.271280, reg: 0.663087, Loss: 23.479323, Acc: 100.00%, SRPQ: 5.152583e-01, Risk: 0.418741
Epoch 1, Iter 3, Loss0: 0.269161, reg: 0.663048, Loss: 23.475835, Acc: 100.00%, SRPQ: 5.152785e-01, Risk: 0.416273
Epoch 1, Iter 4, Loss0: 0.267118, reg: 0.663008, Loss: 23.472388, Acc: 100.00%, SRPQ: 5.152991e-01, Risk: 0.413883
Epoch 1, Iter 5, Loss0: 0.265160, reg: 0.662957, Loss: 23.468667, Acc: 100.00%, SRPQ: 5.153251e-01, Risk: 0.411583
Epoch 1, Iter 6, Loss0: 0.263307, reg: 0.662918, Loss: 23.465428, Acc: 100.00%, SRPQ: 5.153455e-01, Risk: 0.409398
Epoch 1, Iter 7, Loss0: 0.261504, reg: 0.662887, Loss: 23.462566, Acc: 100.00%, SRPQ: 5.153611e-01, Risk: 0.407266
Epoch 1, Iter 8, Loss0: 0.259701, reg: 0.662841, Loss: 23.459150, Acc: 100.00%, SRPQ: 5.153848e-01, Risk: 0.405124
Epoch 1, Iter 9, Loss0: 0.257879, reg: 0.662792, Loss: 23.455594, Acc: 100.00%, 

KeyboardInterrupt: 

In [13]:
params = torch.load("tpcp_1024_w_0.02.pth")
w_value = 0.3
lr = 0.001
clambda = 35

train_tpcp(params, dataloader, lr, clambda, w_value)

Epoch 1, Iter 1, Loss0: 0.213513, reg: 0.430773, Loss: 15.290575, Acc: 100.00%, SRPQ: 6.500063e-01, Risk: 0.347553
Epoch 1, Iter 2, Loss0: 0.208469, reg: 0.430726, Loss: 15.283880, Acc: 100.00%, SRPQ: 6.500370e-01, Risk: 0.340938
Epoch 1, Iter 3, Loss0: 0.203279, reg: 0.430677, Loss: 15.276970, Acc: 100.00%, SRPQ: 6.500689e-01, Risk: 0.334062
Epoch 1, Iter 4, Loss0: 0.198322, reg: 0.430621, Loss: 15.270059, Acc: 100.00%, SRPQ: 6.501052e-01, Risk: 0.327426
Epoch 1, Iter 5, Loss0: 0.193464, reg: 0.430579, Loss: 15.263723, Acc: 100.00%, SRPQ: 6.501327e-01, Risk: 0.320860
Epoch 1, Iter 6, Loss0: 0.188612, reg: 0.430527, Loss: 15.257043, Acc: 100.00%, SRPQ: 6.501666e-01, Risk: 0.314238
Epoch 1, Iter 7, Loss0: 0.183739, reg: 0.430481, Loss: 15.250560, Acc: 100.00%, SRPQ: 6.501965e-01, Risk: 0.307521
Epoch 1, Iter 8, Loss0: 0.178942, reg: 0.430429, Loss: 15.243941, Acc: 100.00%, SRPQ: 6.502304e-01, Risk: 0.300845
Epoch 1, Iter 9, Loss0: 0.174220, reg: 0.430386, Loss: 15.237729, Acc: 100.00%, 

KeyboardInterrupt: 

In [16]:
params = torch.load("tpcp_1024_w_0.3.pth")
w_value = 1
lr = 0.001
clambda = 35

train_tpcp(params, dataloader, lr, clambda, w_value, train_W=False)

Epoch 1, Iter 1, Loss0: 0.132444, reg: -0.000000, Loss: 0.132444, Acc: 100.00%, SRPQ: 1.000000e+00, Risk: 0.232708
Epoch 1, Iter 2, Loss0: 0.126831, reg: 0.000000, Loss: 0.126831, Acc: 100.00%, SRPQ: 1.000000e+00, Risk: 0.224045
Epoch 1, Iter 3, Loss0: 0.120587, reg: 0.000000, Loss: 0.120587, Acc: 100.00%, SRPQ: 1.000000e+00, Risk: 0.214294
Epoch 1, Iter 4, Loss0: 0.115096, reg: 0.000000, Loss: 0.115096, Acc: 100.00%, SRPQ: 1.000000e+00, Risk: 0.205618
Epoch 1, Iter 5, Loss0: 0.109956, reg: 0.000000, Loss: 0.109956, Acc: 100.00%, SRPQ: 1.000000e+00, Risk: 0.197410
Epoch 1, Iter 6, Loss0: 0.104710, reg: 0.000000, Loss: 0.104710, Acc: 100.00%, SRPQ: 1.000000e+00, Risk: 0.188944
Epoch 1, Iter 7, Loss0: 0.099634, reg: 0.000000, Loss: 0.099634, Acc: 100.00%, SRPQ: 1.000000e+00, Risk: 0.180668
Epoch 1, Iter 8, Loss0: 0.094844, reg: 0.000000, Loss: 0.094844, Acc: 100.00%, SRPQ: 1.000000e+00, Risk: 0.172782
Epoch 1, Iter 9, Loss0: 0.090347, reg: 0.000000, Loss: 0.090347, Acc: 100.00%, SRPQ: 1.

KeyboardInterrupt: 